# **Data Processing**

> This notebook performs the data processing workflow used to construct the final analytical dataset for the study. It integrates Facebook and X data collected from multiple sources, standardizes their schemas, validates record consistency, and produces a unified dataset for subsequent analysis.

---

## 1. Import Libraries

> This section imports the libraries required for data preprocessing, merging, validation, and dataset generation. These libraries provide functions for reading Excel files, cleaning and transforming data, handling large numerical identifiers, and performing pattern matching for automated file processing.

In [ ]:
from decimal import Decimal
import pandas as pd

---

## 2. Configure Dataset Directories

> The directory structure used throughout the notebook is configured using relative paths based on the repository structure. This allows the notebook to access the required datasets consistently without modifying individual file locations.

In [ ]:
# Configure dataset directories

DATASET_DIR = "/Dataset"

MISINFORMATION_CASES_DIR = f"{DATASET_DIR}/1. Misinformation Cases"
VIRAL_POSTS_DIR = f"{DATASET_DIR}/2. Viral Posts"

RESHARE_COLLECTION_DIR = f"{DATASET_DIR}/3. Reshare Collection"

APIFY_DIR = f"{DATASET_DIR}/4. Apify"

FB_METADATA_DIR = f"{APIFY_DIR}/fb/1. Scraped Metadata"
X_METADATA_DIR = f"{APIFY_DIR}/x/1. Scraped Metadata"

FB_MERGED_DIR = f"{APIFY_DIR}/fb/2. Merged Scraped Metadata with Reshare Collection"
X_MERGED_DIR = f"{APIFY_DIR}/x/2. Merged Scraped Metadata with Reshare Collection"

FINAL_DATASET_DIR = f"{DATASET_DIR}/5. Final Dataset"

---

## 3. Data Merging

> This section consolidates the raw datasets collected throughout the data acquisition process into a standardized dataset for analysis. Since the Facebook and X datasets were obtained using different scraping workflows and contain different metadata structures, each platform is processed separately before being combined into a single consolidated dataset. Validation procedures are also performed throughout the merging process to ensure record completeness, identifier consistency, and relational integrity.

---
### 3.1 X: Merging Reshare Collection and Scraped Metadata

> Each X misinformation case is processed individually by merging its corresponding reshare collection dataset with the scraped metadata dataset using the shared `post_id` identifier. The reshare collection provides the network structure of the misinformation cascade, while the scraped metadata supplies additional post information such as timestamps, engagement metrics, and URLs. During the merge, column names are standardized, identifiers are preserved as text, timestamps are reformatted into a consistent structure, and only the required variables are retained. The resulting merged dataset for each case is then exported to the merged dataset directory. This procedure is repeated for every X misinformation case included in the study.

> The corresponding datasets for a misinformation case are loaded from the repository. Post identifiers are read as strings to preserve their original values and prevent scientific notation during import.
</br></br> The workflow is parameterized using a CASE_ID variable, allowing the same code to be executed repeatedly for every misinformation case by simply updating the case identifier. Each execution generates one merged dataset in the format (case_id)_x_merged.xlsx, which is later combined with the remaining cases to create the final analytical dataset.

In [ ]:
CASE_ID = 44  # Change this

rows = pd.read_excel(
    f"{X_METADATA_DIR}/{CASE_ID}_x_merged_rows.xlsx",
    dtype=str
)

urls = pd.read_excel(
    f"{X_RESHARE_DIR}/{CASE_ID}_x_url.xlsx",
    dtype=str
)

output_file = (
    f"{x_MERGED_DIR}/{CASE_ID}_fb_merged.xlsx"
)

In [ ]:
# --- CLEAN COLUMN NAMES ---
rows.columns = rows.columns.str.strip()
urls.columns = urls.columns.str.strip()

# --- RENAME COLUMNS IF NEEDED ---
if "tweet_id" in rows.columns:
    rows = rows.rename(columns={"tweet_id": "post_id"})
if "id" in urls.columns:
    urls = urls.rename(columns={"id": "post_id"})
# Rename 'created_at' to 'timestamp'
if "created_at" in rows.columns:
    rows = rows.rename(columns={"created_at": "timestamp"})
if "created_at" in urls.columns:
    urls = urls.rename(columns={"created_at": "timestamp"})

# --- CHECK POST_ID EXISTS ---
if "post_id" not in rows.columns or "post_id" not in urls.columns:
    raise ValueError("Missing 'post_id' column in one of the files")

# --- CONVERT ANY SCIENTIFIC NOTATION TO FULL STRING ---
def fix_post_id(x):
    if pd.isna(x):
        return x
    return str(Decimal(x))

rows["post_id"] = rows["post_id"].apply(fix_post_id)
urls["post_id"] = urls["post_id"].apply(fix_post_id)

# --- KEEP ONLY NECESSARY COLUMNS FROM URLS ---
keep_cols = ["post_id", "misinfo_case_id", "parent_id", "url", "shares", "candidate_category"]
urls = urls[[c for c in keep_cols if c in urls.columns]]

# --- MERGE ---
merged = pd.merge(rows, urls, on="post_id", how="inner", validate="m:1")

# --- CLEAN parent_id ---
if "parent_id" in merged.columns:
    merged["parent_id"] = merged["parent_id"].fillna("N/A").astype(str)
    merged.loc[merged["parent_id"].str.strip() == "", "parent_id"] = "N/A"

# --- FORMAT timestamp ---
if "timestamp" in merged.columns:
    merged["timestamp"] = pd.to_datetime(
        merged["timestamp"], errors="coerce", utc=True
    ).dt.strftime("%d/%m/%Y %I:%M:%S %p")

# --- ADD PLATFORM COLUMN ---
merged["platform"] = "x"

# --- FORCE ALL IDs TO STRING ---
merged["post_id"] = merged["post_id"].astype(str)
if "parent_id" in merged.columns:
    merged["parent_id"] = merged["parent_id"].astype(str)

# --- ORDER COLUMNS TO PREFERRED ---
preferred_order = ["misinfo_case_id", "candidate_category", "post_id", "parent_id", "timestamp", "shares", "url", "platform"]
final_columns = [c for c in preferred_order if c in merged.columns] + [c for c in merged.columns if c not in preferred_order]
merged = merged[final_columns]

# --- SAVE TO EXCEL ---
merged.to_excel(output_file, index=False)
print(f"Saved merged Excel file: {output_file} with {len(merged)} rows")

# --- OPTIONAL: CHECK MATCH COUNT ---
common = set(rows["post_id"]) & set(urls["post_id"])
print("MATCH COUNT:", len(common))

### 3.2 Facebook: Merging Reshare Collection and Scraped Metadata

> Similar to the X workflow, each Facebook misinformation case is processed individually by merging its corresponding reshare collection dataset with the scraped metadata obtained through Apify. Prior to merging, duplicate records are removed from the reshare collection to ensure a unique `post_id` for each entry. The datasets are then standardized by harmonizing column names, formatting timestamps, preserving identifiers as text, and selecting only the variables required for analysis. Additional validation checks are performed to identify unmatched post identifiers and missing parent posts before exporting the merged dataset for each Facebook misinformation case.

> Some Facebook URL collection files contained duplicate `post_id` values due to repeated collection attempts. To prevent duplicate records during merging, duplicate entries were removed while preserving the most recent occurrence of each post.

In [ ]:
CASE_ID = 39  # change this

file_path = (
    f"{RESHARE_COLLECTION_DIR}/fb/{CASE_ID}_fb_url.xlsx"
)

# Read Excel and preserve identifier columns as strings
df = pd.read_excel(
    file_path,
    dtype={
        "post_id": str,
        "parent_id": str
    }
)

# Remove duplicate post IDs while keeping the last occurrence
df_dedup = df.drop_duplicates(
    subset="post_id",
    keep="last"
)

# Overwrite the original file
df_dedup.to_excel(file_path, index=False)

print(f"Duplicates removed. File saved as {file_path}")
print(f"Rows before: {len(df)}")
print(f"Rows after: {len(df_dedup)}")

Duplicates removed. File saved as 39_fb_url.xlsx
Rows before: 1105
Rows after:  921


> The Facebook metadata exported from Apify was merged with the corresponding reshare collection using `post_id` as the common identifier.
</br> </br> During preprocessing, column names were standardized, timestamps were converted into a consistent datetime format, missing parent identifiers were replaced with `"N/A"`, and a platform identifier was added. The final merged dataset follows the same column structure as the X dataset to simplify downstream analysis.

In [ ]:
CASE_ID = 2  # change this

apify = pd.read_excel(
    f"{SCRAPED_METADATA_DIR}/fb/{CASE_ID}_fb_apify.xlsx",
    dtype=str
)

urls = pd.read_excel(
    f"{RESHARE_COLLECTION_DIR}/fb/{CASE_ID}_fb_url.xlsx",
    dtype=str
)

output_file = (
    f"{FB_MERGED_DIR}/{CASE_ID}_fb_merged.xlsx"
)

# Ensure post_id column exists
def ensure_post_id(df):
    if "post_id" in df.columns:
        return df
    if "id" in df.columns:
        return df.rename(columns={"id": "post_id"})
    raise ValueError("Missing 'post_id' (or 'id') column.")

apify = ensure_post_id(apify)
urls = ensure_post_id(urls)

# Standardize column names
rename_map = {
    "create_time": "timestamp",
    "share_count": "shares",
    "post_url": "url"
}

apify = apify.rename(
    columns={c: rename_map[c] for c in apify.columns if c in rename_map}
)

# Keep only the required columns
apify = apify[["post_id", "timestamp", "shares", "url"]]
urls = urls[["post_id", "misinfo_case_id", "candidate_category", "parent_id"]]

# Convert timestamps to a consistent format
if "timestamp" in apify.columns:
    try:
        apify["timestamp"] = pd.to_numeric(
            apify["timestamp"],
            errors="coerce"
        )

        unit = "ms" if apify["timestamp"].max() > 1e12 else "s"

        apify["timestamp"] = pd.to_datetime(
            apify["timestamp"],
            unit=unit,
            errors="coerce"
        ).dt.strftime("%d/%m/%Y %I:%M:%S %p")

    except:
        pass

# Merge the datasets
merged = pd.merge(
    urls,
    apify,
    on="post_id",
    how="left",
    validate="1:1"
)

# Clean parent identifiers
if "parent_id" in merged.columns:
    merged["parent_id"] = merged["parent_id"].fillna("N/A")
    merged.loc[
        merged["parent_id"].str.strip() == "",
        "parent_id"
    ] = "N/A"

# Ensure identifier columns remain as strings
for col in ["post_id", "parent_id"]:
    if col in merged.columns:
        merged[col] = merged[col].astype(str)

# Add platform label
merged["platform"] = "fb"

# Arrange columns
preferred_order = [
    "misinfo_case_id",
    "candidate_category",
    "post_id",
    "parent_id",
    "timestamp",
    "shares",
    "url",
    "platform"
]

final_columns = (
    [c for c in preferred_order if c in merged.columns] +
    [c for c in merged.columns if c not in preferred_order]
)

merged = merged[final_columns]

# Save the merged dataset
merged.to_excel(output_file, index=False)

print(f"Saved merged Excel file: {output_file} ({len(merged)} rows)")

Saved merged Excel file: C:\Users\Ysobella Villariba\Downloads\2_fb_merged.xlsx (rows: 305)


> After merging, the identifiers from both datasets were compared to verify that every post was successfully matched. Any unmatched identifiers were displayed for manual inspection before proceeding to the final combined dataset.

In [ ]:
CASE_ID = 2  # change this

apify = pd.read_excel(
    f"{SCRAPED_METADATA_DIR}/fb/{CASE_ID}_fb_apify.xlsx",
    dtype={"post_id": str}
)

urls = pd.read_excel(
    f"{RESHARE_COLLECTION_DIR}/fb/{CASE_ID}_fb_url.xlsx",
    dtype={"post_id": str}
)

# Compare post IDs between the two datasets
apify_ids = set(apify["post_id"])
url_ids = set(urls["post_id"])

only_in_apify = apify_ids - url_ids
only_in_urls = url_ids - apify_ids

print(f"Post IDs in {CASE_ID}_fb_apify but not in {CASE_ID}_fb_url:")
print(only_in_apify)

print(f"\nPost IDs in {CASE_ID}_fb_url but not in {CASE_ID}_fb_apify:")
print(only_in_urls)

print("\nCount mismatch:")
print("Only in Apify:", len(only_in_apify))
print("Only in URL collection:", len(only_in_urls))
print("Matching IDs:", len(apify_ids & url_ids))

Post IDs in 18_fb_apify but NOT in 18_fb_url:
set()

Post IDs in 18_fb_url but NOT in 18_fb_apify:
set()

Count mismatch:
Only in apify: 0
Only in url: 0


In [ ]:
print("Total Apify IDs:", len(apify_ids))
print("Total URL Collection IDs:", len(url_ids))
print("Matching IDs:", len(apify_ids & url_ids))

Total apify IDs: 305
Total url IDs: 305
Matching IDs: 305


In [ ]:
CASE_ID = 59  # change this

merged = pd.read_excel(
    f"{MERGED_METADATA_DIR}/{CASE_ID}_fb_merged.xlsx",
    engine="openpyxl",
    converters={
        "post_id": str,
        "parent_id": str,
        "shares": int
    }
)

# Identify parent IDs that do not exist as post IDs
parent_ids = merged["parent_id"].dropna()
parent_ids = parent_ids[parent_ids != ""]

post_ids = set(merged["post_id"])
missing_parent_ids = set(parent_ids) - post_ids

# Display the affected misinformation cases
missing_rows = (
    merged.loc[
        merged["parent_id"].isin(missing_parent_ids),
        ["misinfo_case_id", "parent_id"]
    ]
    .drop_duplicates()
    .sort_values("parent_id")
)

print("Missing parent IDs and their corresponding misinformation case IDs:")
print(missing_rows)

print(f"\nTotal unique missing parent IDs: {len(missing_parent_ids)}")

Missing parent_ids and their misinformation_case_id:
Empty DataFrame
Columns: [misinfo_case_id, parent_id]
Index: []

Total unique missing parent_ids: 0


### 3.3 Final Dataset Consolidation

> After all Facebook and X misinformation cases have been processed individually, the resulting merged datasets are consolidated into a single dataset. Each file is automatically sorted according to its misinformation case identifier and platform to maintain a consistent ordering. During consolidation, every record is assigned a corresponding case classification (`fb only`, `x only`, or `cross-platform`) based on the study's predefined misinformation case mapping. The merged datasets are then concatenated into a unified dataset, which serves as the primary input for all succeeding preprocessing, network analysis, amplification metric computation, cross-platform analysis, and clustering procedures.

In [ ]:
# Case type dictionary
case_types = {
    1:"fb only",2:"fb only",3:"fb only",4:"fb only",5:"fb only",
    6:"fb only",7:"fb only",8:"fb only",9:"fb only",10:"fb only",
    11:"fb only",12:"fb only",13:"fb only",14:"fb only",15:"fb only",
    16:"fb only",17:"fb only",18:"fb only",19:"fb only",20:"fb only",
    21:"fb only",22:"fb only",23:"fb only",24:"x only",25:"x only",
    26:"fb only",27:"fb only",28:"fb only",29:"fb only",30:"fb only",
    31:"fb only",32:"fb only",33:"fb only",34:"fb only",35:"fb only",
    36:"fb only",37:"fb only",38:"x only",39:"fb only",40:"fb only",
    41:"fb only",42:"fb only",43:"fb only",44:"cross-platform",
    45:"fb only",46:"fb only",47:"fb only",48:"fb only",
    49:"cross-platform",50:"fb only",51:"fb only",
    52:"cross-platform",53:"fb only",54:"fb only",
    55:"x only",56:"x only",
    57:"cross-platform",58:"cross-platform",
    59:"fb only",60:"x only",
    61:"fb only",62:"fb only",
    63:"cross-platform",64:"fb only",65:"fb only"
}

dfs = []

# Read all merged datasets in case order
for case_id in range(1, 66):

    for platform in ["fb", "x"]:

        file_path = (
            f"{MERGED_DATASET_DIR}/{case_id}_{platform}_merged.xlsx"
        )

        if not os.path.exists(file_path):
            continue

        df = pd.read_excel(
            file_path,
            dtype={
                "post_id": str,
                "parent_id": str
            },
            engine="openpyxl"
        )

        df["post_id"] = df["post_id"].astype(str)
        df["parent_id"] = df["parent_id"].astype(str)

        # Add case type
        df["case_type"] = case_types.get(case_id, "unknown")

        dfs.append(df)

if not dfs:
    raise ValueError("No merged files found.")

# Combine all datasets
merged = pd.concat(dfs, ignore_index=True)

# Save consolidated dataset
output_path = f"{FINAL_DATASET_DIR}/FB_X_Merged.xlsx"
merged.to_excel(output_path, index=False)

print(f"Saved merged dataset: {output_path}")

Files found: ['10_fb_merged.xlsx', '11_fb_merged.xlsx', '12_fb_merged.xlsx', '13_fb_merged.xlsx', '14_fb_merged.xlsx', '15_fb_merged.xlsx', '16_fb_merged.xlsx', '17_fb_merged.xlsx', '18_fb_merged.xlsx', '19_fb_merged.xlsx', '1_fb_merged.xlsx', '20_fb_merged.xlsx', '21_fb_merged.xlsx', '22_fb_merged.xlsx', '23_fb_merged.xlsx', '24_x_merged.xlsx', '25_x_merged.xlsx', '26_fb_merged.xlsx', '27_fb_merged.xlsx', '28_fb_merged.xlsx', '29_fb_merged.xlsx', '2_fb_merged.xlsx', '30_fb_merged.xlsx', '31_fb_merged.xlsx', '32_fb_merged.xlsx', '33_fb_merged.xlsx', '34_fb_merged.xlsx', '35_fb_merged.xlsx', '36_fb_merged.xlsx', '37_fb_merged.xlsx', '38_x_merged.xlsx', '39_fb_merged.xlsx', '3_fb_merged.xlsx', '40_fb_merged.xlsx', '41_fb_merged.xlsx', '42_fb_merged.xlsx', '43_fb_merged.xlsx', '44_fb_merged.xlsx', '44_x_merged.xlsx', '45_fb_merged.xlsx', '46_fb_merged.xlsx', '47_fb_merged.xlsx', '48_fb_merged.xlsx', '49_fb_merged.xlsx', '49_x_merged.xlsx', '4_fb_merged.xlsx', '50_fb_merged.xlsx', '51_fb_m

In [ ]:
import pandas as pd

fb = pd.read_excel(
    "FB_X_Merged.xlsx",
    engine="openpyxl",
    converters={
        "post_id": str,
        "parent_id": str,
        "shares": int
    }
)

# Clean parent_ids
parent_ids = fb["parent_id"].dropna()
parent_ids = parent_ids[parent_ids != ""]

post_ids = set(fb["post_id"])

# Find missing parent_ids
missing_parent_ids = set(parent_ids) - post_ids

# Get rows where parent_id is missing in post_id
missing_rows = fb[fb["parent_id"].isin(missing_parent_ids)][
    ["misinfo_case_id", "parent_id"]
].drop_duplicates().sort_values("parent_id")

print("Missing parent_ids and their misinformation_case_id:")
print(missing_rows)

print("\nTotal unique missing parent_ids:", len(missing_parent_ids))

Missing parent_ids and their misinformation_case_id:
Empty DataFrame
Columns: [misinfo_case_id, parent_id]
Index: []

Total unique missing parent_ids: 0


## 4. Dataset Preprocessing

> The consolidated Facebook and X dataset was loaded and preprocessed to ensure consistency, completeness, and compatibility with the succeeding analytical procedures. This stage involved inspecting the dataset, handling missing values, correcting manually verified records, standardizing timestamp formats, validating chronological consistency, and exporting the finalized dataset.

### 4.1 Load Consolidated Dataset

> The consolidated dataset generated from the previous stage was loaded into memory. Identifier columns were explicitly read as strings to preserve their original values and prevent unintended numeric conversions.

In [ ]:
df = pd.read_excel(
    f"{FINAL_DATASET_DIR}/FB_X_Merged.xlsx",
    engine="openpyxl",
    converters={
        "post_id": str,
        "parent_id": str,
        "shares": int
    }
)

df

### 4.2 Initial Dataset Inspection

> An initial inspection was conducted to verify the dataset dimensions, data types, completeness, and the number of misinformation cases after consolidation. This step ensured that the merged dataset was successfully generated before applying preprocessing operations.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20996 entries, 0 to 20995
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   misinfo_case_id     20996 non-null  object
 1   candidate_category  20996 non-null  object
 2   post_id             20996 non-null  object
 3   parent_id           20925 non-null  object
 4   timestamp           20993 non-null  object
 5   shares              20963 non-null  object
 6   url                 20993 non-null  object
 7   platform            20996 non-null  object
 8   case_type           20996 non-null  object
dtypes: object(9)
memory usage: 1.4+ MB


In [ ]:
df["misinfo_case_id"].nunique()

65

In [ ]:
df["parent_id"].isnull().sum()

71

### 4.3 Handle Missing Values

Missing parent identifiers were replaced with `N/A` to explicitly indicate viral posts that do not originate from another post. Missing share counts were also replaced with zero to ensure numerical consistency during subsequent metric computations.

In [ ]:
df["parent_id"] = df["parent_id"].fillna("N/A")

In [ ]:
df[df["parent_id"] == "N/A"]

### 4.4 Manual Data Corrections

> A small number of records contained incomplete metadata resulting from the data collection process. These entries were manually corrected by referencing the original Facebook posts to restore the missing timestamp, share count, and URL values.

In [ ]:
df[df["timestamp"].isna()]

,misinfo_case_id,candidate_category,post_id,parent_id,timestamp,shares,url,platform,case_type
361,case_3,elective local official,649279140943974,N/A,NaN,NaN,NaN,fb,fb only
978,case_5,senator,24967137842921246,3773722989594084,NaN,NaN,NaN,fb,fb only
3836,case_18,elective local official,9753595244704870,948634050778140,NaN,NaN,NaN,fb,fb only


In [ ]:
df.loc[361, "shares"] = 377
df.loc[361, "timestamp"] = "15/02/2025 15:21:00 PM"
df.loc[361, "url"] = "https://www.facebook.com/BantayEpal/videos/649279140943974/"

df.loc[978, "shares"] = 0
df.loc[978, "timestamp"] = "23/02/2025 4:57:00 PM"
df.loc[978, "url"] = "https://www.facebook.com/rolireyes/posts/pfbid02gfixSNjFUYMAKAafUuuHQ7AYFgAL3fPR8ShmdkWfJTfZi1A3LG2SJ82W4fruj8S1l"

df.loc[3836, "shares"] = 0
df.loc[3836, "timestamp"] = "14/03/2025 4:27:00 PM"
df.loc[3836, "url"] = "https://www.facebook.com/jaynocharli.ca/posts/pfbid02CATeLuMwaqFdib3uhoUQXvcjAQKDdXEYLqJ5HFW7RVxQ3uJcHyK6LXcP3ctVSD2El"

In [ ]:
df[df["timestamp"].isna()]

,misinfo_case_id,candidate_category,post_id,parent_id,timestamp,shares,url,platform,case_type


In [ ]:
df["shares"] = df["shares"].fillna("0")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20996 entries, 0 to 20995
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   misinfo_case_id     20996 non-null  object
 1   candidate_category  20996 non-null  object
 2   post_id             20996 non-null  object
 3   parent_id           20996 non-null  object
 4   timestamp           20996 non-null  object
 5   shares              20996 non-null  object
 6   url                 20996 non-null  object
 7   platform            20996 non-null  object
 8   case_type           20996 non-null  object
dtypes: object(9)
memory usage: 1.4+ MB


### 4.5 Standardize Timestamp Format

> Timestamp values originating from different platforms were converted into a unified datetime format to facilitate chronological analyses, diffusion tracking, and temporal metric computations.

In [ ]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

### 4.6 Validate Parent–Child Chronology

> A validation step was performed to ensure that the temporal ordering of posts was logically consistent. For every misinformation case and platform, the timestamp of the viral post was compared against its corresponding reshare posts. Any reshare occurring earlier than its source post would indicate a data quality issue requiring further investigation.

In [ ]:
problem_groups = []

for (case, platform), group in df.groupby(["misinfo_case_id", "platform"]):

    viral_posts = group[group["parent_id"].isna()]
    shared_posts = group[group["parent_id"].notna()]

    if not viral_posts.empty and not shared_posts.empty:
        earliest_viral = viral_posts["create_time"].min()
        earliest_shared = shared_posts["create_time"].min()

        # If a shared post happened BEFORE the viral post → issue
        if earliest_shared < earliest_viral:
            problem_groups.append(group)

# Display only problematic rows
if problem_groups:
    result = pd.concat(problem_groups).sort_values(["case", "platform", "create_time"])
    print("⚠️ Shared post occurred BEFORE viral post in these case-platform groups:\n")
    display(result)
else:
    print("✅ All case-platform groups are valid. Viral posts were posted before shared posts.")

✅ All case-platform groups are valid. Viral posts were posted before shared posts.


### 4.7 Export Cleaned Dataset

> After preprocessing and validation, the cleaned dataset was exported for use in the subsequent analytical notebook, ensuring that all analyses were performed on a standardized and validated dataset.

In [ ]:
df.to_excel(
    f"{FINAL_DATASET_DIR}/FB_X_Merged.xlsx",
    index=False
)

print("Final consolidated dataset saved successfully.")